# Leaderboard Pipeline 01 — Baseline OOF Diagnostics

This notebook establishes the completed `outputs_dinov3_lora_384` run as the official baseline. Every later experiment must be compared against these OOF metrics and fold-level results.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "leaderboard_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "scripts"))
DATA_ROOT = REPO_ROOT / "BDC2026"
MODEL_OUTPUT = REPO_ROOT / "outputs_dinov3_lora_384"
PIPELINE_ROOT = REPO_ROOT / "leaderboard_pipeline_outputs"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)

LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
DISPLAY_NAMES = {0: "Recyclable", 1: "Electronic", 2: "Organic"}

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("MODEL_OUTPUT:", MODEL_OUTPUT)
print("PIPELINE_ROOT:", PIPELINE_ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

from error_analysis import (
    load_oof, enrich_predictions, build_metrics, confusion_pair_table,
    load_histories, plot_confusion_matrix, plot_training_curves, show_image_grid,
)

STAGE_DIR = PIPELINE_ROOT / "01_baseline"
FIGURES_DIR = STAGE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

oof = enrich_predictions(load_oof(MODEL_OUTPUT, DATA_ROOT))
print("OOF images:", len(oof))
oof.head()


In [ ]:
macro_f1, class_metrics, confusion = build_metrics(oof)
pairs = confusion_pair_table(oof)
print(f"OOF Macro-F1: {macro_f1:.6f}")
display(class_metrics)
display(confusion)
display(pairs.head(10))


In [ ]:
fold_rows = []
for fold, group in oof.groupby("fold"):
    row = {
        "fold": int(fold),
        "images": len(group),
        "accuracy": (group["label"] == group["oof_pred"]).mean(),
        "macro_f1": f1_score(group["label"], group["oof_pred"], average="macro"),
    }
    for label, name in DISPLAY_NAMES.items():
        row[f"f1_{name.lower()}"] = f1_score(group["label"], group["oof_pred"], labels=[label], average="macro", zero_division=0)
    fold_rows.append(row)
fold_metrics = pd.DataFrame(fold_rows).sort_values("fold")
display(fold_metrics)


In [ ]:
class_only = class_metrics[class_metrics["class"].isin(DISPLAY_NAMES.values())].copy()
weakest = class_only.sort_values("f1-score").iloc[0]
print("Weakest class:", weakest["class"], "F1:", weakest["f1-score"])
print("Largest directional confusion:")
display(pairs.head(1))


In [ ]:
confidence_bins = oof.copy()
confidence_bins["bin"] = pd.cut(confidence_bins["confidence"], bins=np.linspace(0, 1, 11), include_lowest=True)
confidence_summary = confidence_bins.groupby("bin", observed=False).agg(
    images=("path", "size"),
    accuracy=("correct", "mean"),
    mean_confidence=("confidence", "mean"),
    error_rate=("correct", lambda x: 1-x.mean()),
).reset_index()
display(confidence_summary)

plt.figure(figsize=(7,5))
valid = confidence_summary.dropna(subset=["mean_confidence", "accuracy"])
plt.plot(valid["mean_confidence"], valid["accuracy"], marker="o", label="OOF")
plt.plot([0,1], [0,1], linestyle="--", label="perfect calibration")
plt.xlabel("Mean confidence"); plt.ylabel("Accuracy"); plt.title("OOF calibration")
plt.grid(alpha=.25); plt.legend(); plt.show()


In [ ]:
history = load_histories(MODEL_OUTPUT)
plot_confusion_matrix(confusion, FIGURES_DIR / "confusion_matrix.png")
plot_training_curves(history, FIGURES_DIR)
show_image_grid(
    oof[~oof["correct"]], n=30, sort_by="confidence", ascending=False,
    title="Highest-confidence baseline errors",
    save_path=FIGURES_DIR / "highest_confidence_errors.png",
)


In [ ]:
summary = {
    "images": len(oof),
    "macro_f1": float(macro_f1),
    "accuracy": float(oof["correct"].mean()),
    "wrong_predictions": int((~oof["correct"]).sum()),
    "weakest_class": weakest["class"],
    "weakest_class_f1": float(weakest["f1-score"]),
    "mean_fold_macro_f1": float(fold_metrics["macro_f1"].mean()),
    "std_fold_macro_f1": float(fold_metrics["macro_f1"].std(ddof=0)),
}
oof.to_csv(STAGE_DIR / "baseline_oof_enriched.csv", index=False)
class_metrics.to_csv(STAGE_DIR / "per_class_metrics.csv", index=False)
confusion.to_csv(STAGE_DIR / "confusion_matrix.csv")
pairs.to_csv(STAGE_DIR / "confusion_pairs.csv", index=False)
fold_metrics.to_csv(STAGE_DIR / "fold_metrics.csv", index=False)
confidence_summary.to_csv(STAGE_DIR / "confidence_calibration.csv", index=False)
(STAGE_DIR / "baseline_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary


## Gate interpretation

The weakest class and largest confusion pair determine what to clean and what model diversity to add. Do not judge the next experiment only by overall OOF Macro-F1; require fold stability and no meaningful degradation in the weakest class.
